# Job skill tag audit

Run from this folder. The notebook applies the documented rubric in `writeup.md`, normalizes clear aliases, produces per-description labels, and reports aggregate counts.

In [ ]:
from pathlib import Path
import pandas as pd, re
ROOT=Path.cwd()
jobs=pd.read_csv(ROOT/"job-descriptions-tagged.csv").fillna("")
skills=pd.read_csv(ROOT/"skill-list.csv").skill.tolist()
aliases={"Financial Modelling":["Financial Modeling"],"Node.js":["Node"],"JavaScript":["JS"],"Microsoft Excel":["Excel"],"PostgreSQL":["Postgres"],"REST APIs":["REST API"],"Machine Learning":["ML"]}
patterns={s:[s]+aliases.get(s,[]) for s in skills}
optional_re=re.compile(r"\b(plus|not a must|bonus|nice to have|optional)\b",re.I)
def extract_expected(desc):
    out={}
    for skill,variants in patterns.items():
        for v in sorted(variants,key=len,reverse=True):
            for m in re.finditer((r"(?<![A-Za-z0-9.])"+re.escape(v)+r"(?![A-Za-z0-9])" if v=="JS" else r"(?<![A-Za-z0-9])"+re.escape(v)+r"(?![A-Za-z0-9])"),desc,re.I):
                lo=max(0,desc.rfind(".",0,m.start())+1); ends=[p for p in (desc.find(".",m.end()),desc.find(";",m.end())) if p>=0]; hi=min(ends) if ends else len(desc)
                out[skill]="optional" if optional_re.search(desc[lo:hi]) else "required"; break
            if skill in out: break
    if "User Research" in out and not re.search(r"(?<![A-Za-z0-9])Research(?![A-Za-z0-9])", re.sub(r"(?i)User Research","",desc)):
        out.pop("Research",None)
    return out
def norm(tag):
    t=re.sub(r"\s+"," ",str(tag).strip()).casefold()
    for canonical,variants in patterns.items():
        if t==canonical.casefold() or any(t==v.casefold() for v in variants): return canonical
    return None
rows=[]
for _,r in jobs.iterrows():
    expected=extract_expected(r.description); pred=[x.strip() for x in str(r.model_tags).split(";") if x.strip()]; right=[]; wrong=[]; seen=set()
    for tag in pred:
        c=norm(tag)
        if c and c in expected and c not in seen: right.append(tag); seen.add(c)
        else: wrong.append(tag)
    missing=[s for s,k in expected.items() if k=="required" and s not in seen]
    rows.append({"jd_id":r.jd_id,"title":r.title,"model_tags":r.model_tags,"required_skills":"; ".join(s for s,k in expected.items() if k=="required"),"optional_skills":"; ".join(s for s,k in expected.items() if k=="optional"),"right_tags":"; ".join(right),"wrong_tags":"; ".join(wrong),"missing_required_skills":"; ".join(missing),"tag_count":len(pred),"right_count":len(right),"wrong_count":len(wrong),"missed_required_count":len(missing)})
labels=pd.DataFrame(rows); labels.to_csv(ROOT/"labels.csv",index=False)
nt=int(labels.tag_count.sum()); nr=int(labels.right_count.sum()); nw=int(labels.wrong_count.sum()); nm=int(labels.missed_required_count.sum()); required=sum(sum(v=="required" for v in extract_expected(r.description).values()) for _,r in jobs.iterrows())
print(f"Descriptions: {len(labels)} | predicted tags: {nt} | right: {nr} ({nr/nt:.1%}) | wrong: {nw} ({nw/nt:.1%})")
print(f"Required skill mentions: {required} | missed: {nm} ({nm/required:.1%})")
labels.head()